# Filter data: HK1-R12Ter
## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from itertools import product

from hk1_r12ter_analysis.config import INTERIM_DATA_DIR, logger
from hk1_r12ter_analysis.data.filtering import (
    filter_low_abundance,
    filter_low_quality_by_missingness,
    filter_low_variance,
)
from hk1_r12ter_analysis.io import (
    load_dataframe_from_file,
    make_filename,
    save_dataframe_to_file,
)

## Load data and filter

In [ ]:
sample_key = "Sample"
group_key = "Group"
source_types = ["RBC", "Plasma"]
data_types = ["Proteins", "Metabolites", "Lipids", "Oxylipins"]
value_type = "Intensities"
# Low-quality filtering
missingness_percent = None
use_groupwise_threshold = True

# Low-variance filtering
low_variance_method = "iqr"
low_variance_percent = 0

# Low-abundance filtering
low_abundance_method = None
low_abundance_percent = 0

# DataFrame format axis=0: (Samples, Features) or axis=1: (Features, Samples)
axis = 0

# Directory paths
input_dirpath = INTERIM_DATA_DIR / "HK1-R12Ter" / "Formatted"
output_dirpath = INTERIM_DATA_DIR / "HK1-R12Ter" / "Filtered"
output_dirpath.mkdir(parents=True, exist_ok=True)

### Filter

In [ ]:
index_cols = [key for key in [sample_key, group_key] if key]
for source_type, data_type in product(source_types, data_types):
    logger.debug(f"Processing data for '{source_type}-{data_type}'...")
    # Load data
    filename_args = (source_type, data_type, value_type)
    filename = make_filename(*filename_args)
    df = load_dataframe_from_file(input_dirpath / filename, index_col=index_cols)
    # Apply low-quality filter
    if missingness_percent not in {None, 0, 1}:
        df = filter_low_quality_by_missingness(
            df,
            percent=missingness_percent,
            use_groupwise_threshold=use_groupwise_threshold,
            group_key=group_key,
            axis=axis,
        )
    # TODO Apply low-repeatability filter
    # Apply low variance filter
    if low_variance_method is not None:
        df = filter_low_variance(
            df, method=low_variance_method, percent=low_variance_percent, axis=axis
        )
    # Apply low-abundance filter
    if low_abundance_method is not None:
        df = filter_low_abundance(
            df, method=low_abundance_method, percent=low_abundance_percent, axis=axis
        )

    save_dataframe_to_file(df, output_dirpath / filename, index=True)
    logger.info(f"Processed data for '{source_type}-{data_type}'.")